可以在工具或中间件中访问长期记忆

# 何时写入记忆
官方介绍了两种方式。

## 1. 在主流程里写（hot path）
也就是：用户发消息，AI一边回答，一边决定要不要记下来。

**优点：**
- 立即生效
- 下一轮马上能用
- 用户可感知，透明

**缺点：**
- 增加延迟
- 逻辑变复杂

## 2. 在后台写（background）
就是先回答用户，记忆整理放到后台**异步**做。

**优点：**
- 主流程更快
- 记忆逻辑更独立
- 更适合批量整理

**缺点：**
- 不能立刻生效
- 要决定多久整理一次
- 触发时机不好选

**工程上通常这么选：**
- 用户偏好、账号资料：可热路径写
- 对话摘要、经验沉淀、行为分析：更适合后台写

# 在工具中访问

## 示例1-InMemoryStore

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

from langchain_core.messages import HumanMessage
from typing import NotRequired

from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

class CustomState(AgentState):
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """
    将用户信息保存在长期记忆中

    Args:
        name: 用户名

    Returns:
        str: 保存状态
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """
    从长期记忆中读取用户信息

    Returns:
        str: 用户信息
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item else "unknown"

agent = create_agent(
    model=model,
    tools=[save_user_info, get_user_info],
    store=store,
    system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
    state_schema=CustomState,
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()


============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_8QP70jxR2tgdeiFgkz3mC6CX)
 Call ID: call_8QP70jxR2tgdeiFgkz3mC6CX
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好，小花，很高兴认识你！我会记住你的名字。
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_leev2c13lRdUVch5Gvp6zVbW)
 Call ID: call_leev2c13lRdUVch5Gvp6zVbW
  Args:
================================= Tool Message ========

## 示例2-PostgreSQLStore

In [ ]:
from typing import NotRequired

from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.store.postgres import PostgresStore

DB_URL = os.getenv("DATABASE_URL")

class CustomState(AgentState):
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """
    将用户信息保存在长期记忆中

    Args:
        name: 用户名

    Returns:
        str: 保存状态
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """
    从长期记忆中读取用户信息

    Returns:
        str: 用户信息
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item else "unknown"

with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    agent = create_agent(
        model=model,
        tools=[save_user_info, get_user_info],
        store=store,
        system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
        state_schema=CustomState,
    )

    print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
    response1 = agent.invoke({
        "messages": "你好，很高兴认识你，我是小花",
        "user_id": "user-1"
    })
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
    response2 = agent.invoke({
        "messages": "我是谁",
        "user_id": "user-1"
    })
    for msg in response2["messages"]:
        msg.pretty_print()


# 在中间件中访问

## Node‑style hooks中访问
以 `before_model` 为例，其钩子函数签名如下
```python
def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
```

**Runtime 定义如下**
```python
@dataclass(**_DC_KWARGS)
class Runtime(Generic[ContextT]):
    context: ContextT = field(default=None)  # type: ignore[assignment]
    """Static context for the graph run, like `user_id`, `db_conn`, etc.

    Can also be thought of as 'run dependencies'."""

    store: BaseStore | None = field(default=None)
    """Store for the graph run, enabling persistence and memory."""

    stream_writer: StreamWriter = field(default=no_op_stream_writer)
    """Function that writes to the custom stream."""

    previous: Any = field(default=None)
    """The previous return value for the given thread.

    Only available with the functional API when a checkpoint is provided.
    """
    ...
```

所以，我们可以通过 `runtime.store` 在中间件中访问长期记忆。

### Wrap‑style hooks中访问
#### 1. wrap_model_call
钩子函数签名如下
```python
def wrap_model_call(
    self,
    request: ModelRequest[ContextT],
    handler: Callable[[ModelRequest[ContextT]], ModelResponse[ResponseT]],
) -> ModelResponse[ResponseT] | AIMessage | ExtendedModelResponse[ResponseT]:
```

**ModelRequest 定义如下**
```python
@dataclass(init=False)
class ModelRequest(Generic[ContextT]):
    """Model request information for the agent.

    Type Parameters:
        ContextT: The type of the runtime context. Defaults to `None` if not specified.
    """

    model: BaseChatModel
    messages: list[AnyMessage]  # excluding system message
    system_message: SystemMessage | None
    tool_choice: Any | None
    tools: list[BaseTool | dict[str, Any]]
    response_format: ResponseFormat[Any] | None
    state: AgentState[Any]
    runtime: Runtime[ContextT]
    model_settings: dict[str, Any] = field(default_factory=dict)
    ...
```

所以，可以通过 `request.runtime.store` 访问长期记忆。

#### 2. wrap_tool_call
钩子函数签名如下
```python
def wrap_tool_call(
    self,
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
) -> ToolMessage | Command[Any]:
```

**ToolCallRequest 定义如下**
```python
@dataclass
class ToolCallRequest:
    """Tool execution request passed to tool call interceptors.

    Attributes:
        tool_call: Tool call dict with name, args, and id from model output.
        tool: BaseTool instance to be invoked, or None if tool is not
            registered with the `ToolNode`. When tool is `None`,
            interceptors can handle the request without validation. If the interceptor calls
            `execute()`, validation will occur and raise an error for unregistered tools.
        state: Agent state (`dict`, `list`, or `BaseModel`).
        runtime: LangGraph runtime context (optional, `None` if outside graph).
    """

    tool_call: ToolCall
    tool: BaseTool | None
    state: Any
    runtime: ToolRuntime
    ...
```

**ToolRuntime 定义如下**
```python
@dataclass
class ToolRuntime(_DirectlyInjectedToolArg, Generic[ContextT, StateT]):
    state: StateT
    context: ContextT
    config: RunnableConfig
    stream_writer: StreamWriter
    tool_call_id: str | None
    store: BaseStore | None
```

所以，可以通过 `request.runtime.store` 访问长期记忆。


# 中间件访问长期记忆示例

下面完全使用 `InMemoryStore`，不依赖 PostgreSQL。为了让本节只关注 Store，用户 ID 继续沿用前文的 `CustomState.user_id`：

- `state["user_id"]` 或 `request.state["user_id"]`：确定访问哪个用户的 namespace。
- `runtime.store` 或 `request.runtime.store`：访问 Agent 创建时注入的长期记忆 Store。

这里先手动写入两位用户的画像，使后面的 Middleware 行为可预测，不依赖模型是否主动调用保存工具。

In [ ]:
from langgraph.store.memory import InMemoryStore


middleware_store = InMemoryStore()

profiles = {
    "user-1": {"name": "小花", "drink": "拿铁", "style": "简洁"},
    "user-2": {"name": "小明", "drink": "绿茶", "style": "详细"},
}

for user_id, profile in profiles.items():
    middleware_store.put(
        ("users", user_id, "profile"),
        "main",
        profile,
    )


## 示例3：在 `before_model` 中读写 Store

`before_model` 在每次模型调用前执行。下面的 Middleware 读取当前用户之前的模型调用次数，加一后再写回 Store。

注意：它统计的是**模型节点执行次数**，不是对话轮数。带工具的 Agent 在一次 `invoke()` 中可能多次进入模型节点，因此计数可能增加多次。

In [ ]:
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime


@before_model
def count_model_calls(
    state: CustomState,
    runtime: Runtime,
) -> None:
    store = runtime.store
    if store is None:
        raise RuntimeError("Agent 创建时必须传入 store")

    user_id = state["user_id"]
    namespace = ("users", user_id, "metrics")
    old_item = store.get(namespace, "model_calls")
    old_count = old_item.value["count"] if old_item else 0

    store.put(namespace, "model_calls", {"count": old_count + 1})
    print(f"[before_model] {user_id} 的模型调用次数：{old_count + 1}")


## 示例4：在 `wrap_model_call` 中读取记忆并注入提示词

`wrap_model_call` 可以修改本次真正发送给模型的请求。下面根据 `user_id` 从 Store 读取用户画像，并通过 `request.override(system_message=...)` 临时扩展 system prompt。

这种修改只影响当前模型调用，不会把用户画像追加进 Agent 的消息历史，适合把偏好、权限或业务配置作为动态上下文注入。

In [ ]:
from collections.abc import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.messages import SystemMessage


@wrap_model_call
def inject_user_memory(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    store = request.runtime.store
    if store is None:
        raise RuntimeError("Agent 创建时必须传入 store")

    user_id = request.state["user_id"]
    item = store.get(("users", user_id, "profile"), "main")
    profile = item.value if item else {}

    base_prompt = (
        request.system_message.content
        if request.system_message is not None
        else ""
    )
    memory_prompt = (
        f"{base_prompt}\n\n"
        f"当前用户的长期记忆：{profile}\n"
        "回答时自然使用这些信息，不要声称自己调用了数据库。"
    )

    new_request = request.override(
        system_message=SystemMessage(content=memory_prompt)
    )
    return handler(new_request)


memory_agent = create_agent(
    model=model,
    tools=[],
    store=middleware_store,
    middleware=[count_model_calls, inject_user_memory],
    system_prompt="你是了解用户偏好的个人助手。",
    state_schema=CustomState,
)


## 使用同一个 Agent 服务不同用户

两次调用使用相同的 Agent 和 Store，只改变输入 State 中的 `user_id`。Middleware 会自动访问不同 namespace，因此模型应分别回答小花和小明的偏好。

In [ ]:
for user_id in ("user-1", "user-2"):
    result = memory_agent.invoke(
        {
            "messages": "我叫什么名字，喜欢喝什么？请按我喜欢的风格回答。",
            "user_id": user_id,
        }
    )
    print(f"\n[{user_id}] {result['messages'][-1].content}")


## 查看 Middleware 写入的长期记忆

下面直接读取 Store，验证 `before_model` 的写入结果。由于当前 Agent 没有工具循环，每次 `invoke()` 通常只调用一次模型，因此两位用户的计数通常都是 1。

`InMemoryStore` 只在当前 Python 进程和当前对象存活期间保留数据：它可以演示跨多次 Agent 调用共享长期记忆的接口语义，但重启 Notebook 内核后数据会丢失。生产环境仍应换成 PostgreSQL 等持久化 Store。

In [ ]:
for user_id in ("user-1", "user-2"):
    profile = middleware_store.get(("users", user_id, "profile"), "main")
    metrics = middleware_store.get(("users", user_id, "metrics"), "model_calls")
    print(user_id, "profile=", profile.value, "metrics=", metrics.value)
